# Integrated Gradients on the NASA held-out test set

This notebook recomputes the manuscript attribution summaries from the 187 cycle-level Integrated Gradients records. The records were produced with the primary NASA checkpoint, a 50-step Gauss–Legendre quadrature, a training-set mean voltage and current baseline, temperature baseline $25^\circ\mathrm{C}$, and normalized-time baseline $0$.

Attributions measure the sensitivity of the trained model to its inputs. They are not measurements of SEI thickness, lithium deposition, or reaction rates. Temperature attribution differs by cell and should not be read as a single universal temperature effect. This notebook does not use SHAP.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

def repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        marker = candidate / "data" / "nasa_manuscript" / "explainability" / "per_sample_attributions.csv"
        if marker.exists():
            return candidate
    raise FileNotFoundError("Run this notebook from the TE-Q-Transformer repository.")

ROOT = repo_root()
IG_DIR = ROOT / "data" / "nasa_manuscript" / "explainability"
samples = pd.read_csv(IG_DIR / "per_sample_attributions.csv")
print("Repository root:", ROOT)
print("Cycles:", len(samples), "cells:", samples.groupby("cell_id").size().to_dict())
if len(samples) != 187:
    raise RuntimeError(f"Expected 187 cycles, found {len(samples)}")


Repository root: E:\TE-Q-Transformer\TE-Q-Transformer
Cycles: 187 cells: {'B0018': 132, 'B0032': 39, 'B0053_test': 16}


## Completeness and global shares


In [2]:
channels = ["voltage", "current", "temperature", "time"]
labels = ["Voltage", "Current", "Temperature", "Normalized time"]
abs_cols = [f"attr_abs_{c}" for c in channels]
completeness = samples["completeness_error"].to_numpy(dtype=float)
print(f"Maximum completeness error: {completeness.max():.6e}")
print(f"Mean completeness error: {completeness.mean():.6e}")
total_abs = samples[abs_cols].sum(axis=0).to_numpy()
sample_weighted = 100.0 * total_abs / total_abs.sum()
print("Sample-weighted shares (%):")
for name, value in zip(labels, sample_weighted):
    print(f"  {name}: {value:.2f}")
cell_rows = []
for cell, group in samples.groupby("cell_id"):
    totals = group[abs_cols].sum(axis=0).to_numpy()
    shares = 100.0 * totals / totals.sum()
    cell_rows.append({"cell": cell, "n": len(group), **dict(zip(["Voltage", "Current", "Temperature", "Time"], shares))})
cell_table = pd.DataFrame(cell_rows).sort_values("cell")
equal_cell = cell_table[["Voltage", "Current", "Temperature", "Time"]].mean(axis=0)
print("Equal-cell shares (%):")
print(equal_cell.round(2).to_string())
print("Cell temperature shares (%):")
print(cell_table[["cell", "n", "Temperature"]].to_string(index=False))


Maximum completeness error: 6.407499e-07
Mean completeness error: 2.857318e-07
Sample-weighted shares (%):
  Voltage: 19.31
  Current: 44.06
  Temperature: 1.93
  Normalized time: 34.69
Equal-cell shares (%):
Voltage        25.09
Current        37.20
Temperature     3.55
Time           34.16
Cell temperature shares (%):
      cell   n  Temperature
     B0018 132     0.001873
     B0032  39     0.212023
B0053_test  16    10.437877


## Temperature replacement and permutation


In [3]:
ablation = pd.read_csv(IG_DIR / "channel_ablation_results.csv")
temperature = ablation[ablation["Channel"] == "Temperature"].iloc[0]
print("Temperature channel replaced by 25 C, frozen model:")
print(f"  B0018 RMSE: {temperature['B0018_RMSE']:.4f}")
print(f"  B0032 RMSE: {temperature['B0032_RMSE']:.4f}")
print(f"  B0053 test RMSE: {temperature['B0053_test_RMSE']:.4f}")
print(f"  Macro RMSE: {temperature['Macro_RMSE']:.4f}")
permutation = pd.read_csv(IG_DIR / "permutation_importance_results.csv")
print("Permutation pooled RMSE = stored base RMSE + mean delta RMSE")
for _, row in permutation.iterrows():
    pooled = float(row["Pooled_Base_RMSE"] + row["Mean_Delta_RMSE"])
    print(f"  {row['Channel']}: {pooled:.4f} (delta {row['Mean_Delta_RMSE']:.4f})")


Temperature channel replaced by 25 C, frozen model:
  B0018 RMSE: 0.0271
  B0032 RMSE: 0.0121
  B0053 test RMSE: 0.1657
  Macro RMSE: 0.0683
Permutation pooled RMSE = stored base RMSE + mean delta RMSE
  Voltage: 0.1253 (delta 0.1016)
  Current: 0.1835 (delta 0.1599)
  Temperature: 0.0543 (delta 0.0306)
  Time: 0.0237 (delta 0.0000)


## Temperature attribution and SOH


In [4]:
from scipy import stats
for cell, group in samples.groupby("cell_id"):
    rho, p_value = stats.spearmanr(group["ground_truth_soh"], group["attr_abs_temperature"])
    print(f"{cell}: Spearman rho = {rho:.3f}, p = {p_value:.3e}, n = {len(group)}")
bins = pd.read_csv(IG_DIR / "temperature_vs_soh_bins.csv")
print(bins.to_string(index=False))


B0018: Spearman rho = -0.758, p = 6.455e-26, n = 132
B0032: Spearman rho = 0.602, p = 4.989e-05, n = 39
B0053_test: Spearman rho = 0.541, p = 3.040e-02, n = 16
     SOH_Bin  Sample_Count            Cells_Present  Mean_SOH  Mean_Abs_Temp_Attr  Std_Abs_Temp_Attr  Mean_Pct_Temp_Attr  Std_Pct_Temp_Attr
[0.70, 0.80)            58                    B0018  0.759088            0.000011       4.178338e-07            0.002370           0.000106
[0.80, 0.90)            36                    B0018  0.856680            0.000010       1.169619e-06            0.001934           0.000233
[0.90, 1.00)            62 B0018, B0032, B0053_test  0.953406            0.044398       8.176061e-02            2.367675           4.367911
[1.00, 1.15]            31 B0018, B0032, B0053_test  1.040922            0.017177       5.043052e-02            0.934432           2.652584


## Gauss–Legendre check on the primary checkpoint, when the checkpoint is present


In [5]:
ckpt = ROOT / "artifacts" / "nasa" / "01_rich_entangler" / "nasa_teq_transformer_best.pth"
nasa_dir = ROOT / "datasets" / "NASA" / "processed"
if not ckpt.exists() or not nasa_dir.exists():
    print("Checkpoint or processed NASA arrays are not in this checkout.")
    print("The tables above still use the committed 187-cycle Integrated Gradients records from that checkpoint.")
else:
    import sys
    sys.path.insert(0, str(ROOT))
    import torch
    from models.proposed.te_q_transformer import TEQTransformer, rich_entangler_config
    from src.data.nasa_loader import (
        load_full_cell, split_cell_70_30, transform_with_scaler, fit_train_scaler,
        NASA_TRAIN_CELLS, NASA_SPLIT_CELL,
    )

    state = torch.load(ckpt, map_location="cpu", weights_only=False)
    model = TEQTransformer(rich_entangler_config())
    model.load_state_dict(state)
    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    print("Loaded primary checkpoint. Ea_sei =", float(model.quantum_embed.Ea_sei), "Ea_pl =", float(model.quantum_embed.Ea_pl))

    def gauss_legendre_ig(batch_x, baseline, steps=50):
        nodes, weights = np.polynomial.legendre.leggauss(steps)
        alphas = torch.tensor(0.5 * (nodes + 1.0), dtype=torch.float32)
        quad_weights = torch.tensor(0.5 * weights, dtype=torch.float32)
        interpolated = baseline + alphas.view(-1, 1, 1) * (batch_x - baseline)
        interpolated = interpolated.clone().detach().requires_grad_(True)
        predictions = model(interpolated)
        gradients = torch.autograd.grad(predictions.sum(), interpolated)[0]
        weighted = (gradients * quad_weights.view(-1, 1, 1)).sum(dim=0, keepdim=True)
        return (batch_x - baseline) * weighted

    train_parts = [load_full_cell(nasa_dir, cell_id)[0] for cell_id in NASA_TRAIN_CELLS]
    early_x, _ = split_cell_70_30(nasa_dir, NASA_SPLIT_CELL)[0]
    train_parts.append(early_x)
    train_x = torch.cat(train_parts, dim=0)
    scaler = fit_train_scaler(train_x)
    x_b18, _ = load_full_cell(nasa_dir, "B0018")
    x_tensor = transform_with_scaler(x_b18[:1], scaler, clip=True)
    train_scaled = transform_with_scaler(train_x, scaler, clip=True)
    baseline = torch.zeros(1, 512, 4)
    baseline[0, :, 0] = train_scaled[:, :, 0].mean(dim=0)
    baseline[0, :, 1] = train_scaled[:, :, 1].mean(dim=0)
    baseline[0, :, 2] = 25.0
    baseline[0, :, 3] = 0.0
    with torch.no_grad():
        f_x = model(x_tensor).item()
        f_0 = model(baseline).item()
    attribution = gauss_legendre_ig(x_tensor, baseline, steps=50)
    completeness_error = abs((f_x - f_0) - float(attribution.sum()))
    print(f"B0018 first cycle completeness error: {completeness_error:.6e}")
    print("This single-cycle check uses the same quadrature as the stored 187-cycle table.")


E:\TE-Q-Transformer\TE-Q-Transformer\models\proposed\te_q_transformer.py:313: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=self.cfg.n_layers)


Loaded primary checkpoint. Ea_sei = 2.1036643981933594 Ea_pl = 2.620927095413208


B0018 first cycle completeness error: 4.097819e-07
This single-cycle check uses the same quadrature as the stored 187-cycle table.


## Interpretation

Sample-weighted shares give more influence to B0018 because that cell supplies 132 of the 187 cycles. Equal-cell shares average the three test settings. Temperature attribution is about 0.002% on B0018, about 0.21% on B0032, and about 10.4% on the B0053 test portion. Replacing the $4^\circ\mathrm{C}$ input with $25^\circ\mathrm{C}$ changes the B0053 test error sharply, while the same replacement leaves B0018 nearly unchanged. That is input sensitivity of this checkpoint, not a physical assay of degradation.
